# SegFormer Polyp Segmentation - Kaggle Training

This notebook trains a SegFormer model for polyp segmentation on the Kvasir-SEG dataset.

## 1. Setup & Dependencies

In [ ]:
!pip install -q torch transformers accelerate opencv-python albumentations scikit-learn tqdm numpy Pillow requests

In [ ]:
import os
import json
import torch
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
class Config:
    # Paths
    RAW_DATA_DIR = "/kaggle/input/kvasir-seg"  # Update with your Kaggle dataset path
    IMAGE_DIR = os.path.join(RAW_DATA_DIR, "images")
    MASK_DIR = os.path.join(RAW_DATA_DIR, "masks")
    OUTPUT_DIR = "/kaggle/working/output"
    CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
    SPLIT_JSON = os.path.join(OUTPUT_DIR, "dataset_split.json")
    
    # Model
    MODEL_NAME = "nvidia/mit-b0"
    IMG_SIZE = 512
    NUM_CLASSES = 2  # background + polyp
    
    # Training
    BATCH_SIZE = 8
    LEARNING_RATE = 1e-4
    EPOCHS = 50
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Create directories
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Device: {Config.DEVICE}")
print(f"Image directory: {Config.IMAGE_DIR}")
print(f"Mask directory: {Config.MASK_DIR}")

## 3. Dataset & Transforms

In [ ]:
def get_transforms(img_size=512):
    train_transform = A.Compose(
        [
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]
    )

    val_transform = A.Compose(
        [
            A.Resize(img_size, img_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ]
    )

    return train_transform, val_transform


class KvasirSegDataset(Dataset):
    """Dataset for Kvasir-SEG polyp segmentation."""

    def __init__(self, file_ids, image_dir, mask_dir, transform=None):
        self.file_ids = list(file_ids)
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        file_id = self.file_ids[idx]
        image_path = os.path.join(self.image_dir, f"{file_id}.jpg")
        mask_path = os.path.join(self.mask_dir, f"{file_id}.jpg")

        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")

        mask = (mask > 127).astype("float32")

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        return {
            "pixel_values": image,
            "labels": torch.as_tensor(mask, dtype=torch.long),
            "id": file_id,
        }

print("Dataset class defined.")

## 4. Create Dataset Split

In [ ]:
# Get all image files
image_files = sorted([f.replace('.jpg', '') for f in os.listdir(Config.IMAGE_DIR) if f.endswith('.jpg')])
print(f"Total images: {len(image_files)}")

# Create train/val split (80/20)
np.random.seed(42)
indices = np.random.permutation(len(image_files))
split_idx = int(0.8 * len(image_files))

train_ids = [image_files[i] for i in indices[:split_idx]]
val_ids = [image_files[i] for i in indices[split_idx:]]

splits = {
    "train": train_ids,
    "val": val_ids
}

# Save split
with open(Config.SPLIT_JSON, 'w') as f:
    json.dump(splits, f)

print(f"Train samples: {len(train_ids)}")
print(f"Val samples: {len(val_ids)}")

## 5. Metrics

In [ ]:
def dice_score(pred, target, smooth=1e-6):
    """Calculate Dice coefficient."""
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice


def iou_score(pred, target, smooth=1e-6):
    """Calculate Intersection over Union."""
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    intersection = (pred * target).sum()
    union = (pred + target - pred * target).sum()
    
    iou = (intersection + smooth) / (union + smooth)
    return iou

print("Metrics defined.")

## 6. Load Data

In [ ]:
train_tf, val_tf = get_transforms(img_size=Config.IMG_SIZE)

train_ds = KvasirSegDataset(splits["train"], Config.IMAGE_DIR, Config.MASK_DIR, transform=train_tf)
val_ds = KvasirSegDataset(splits["val"], Config.IMAGE_DIR, Config.MASK_DIR, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 7. Initialize Model

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(
    Config.MODEL_NAME,
    num_labels=Config.NUM_CLASSES,
    id2label={0: "background", 1: "polyp"},
    label2id={"background": 0, "polyp": 1},
    ignore_mismatched_sizes=True,
)
model.to(Config.DEVICE)

optimizer = AdamW(model.parameters(), lr=Config.LEARNING_RATE)

print(f"Model loaded: {Config.MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 8. Training Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    total_dice = 0

    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values, labels=labels)

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        upsampled_logits = torch.nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
        )

        dice = dice_score(upsampled_logits, labels)
        total_loss += loss.item()
        total_dice += dice.item()

        pbar.set_postfix({"Loss": loss.item(), "Dice": dice.item()})

    return total_loss / len(loader), total_dice / len(loader)


def validate(model, loader, device):
    model.eval()
    total_dice = 0
    total_iou = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits

            upsampled_logits = torch.nn.functional.interpolate(
                logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
            )

            total_dice += dice_score(upsampled_logits, labels).item()
            total_iou += iou_score(upsampled_logits, labels).item()

    return total_dice / len(loader), total_iou / len(loader)

print("Training functions defined.")

## 9. Training Loop

In [ ]:
best_dice = 0.0
history = []

for epoch in range(Config.EPOCHS):
    print(f"\nEpoch {epoch + 1}/{Config.EPOCHS}")

    train_loss, train_dice = train_one_epoch(model, train_loader, optimizer, Config.DEVICE)
    val_dice, val_iou = validate(model, val_loader, Config.DEVICE)

    print(f"Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
    print(f"Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        best_path = os.path.join(Config.CHECKPOINT_DIR, "segformer_polyp_best.pth")
        torch.save(model.state_dict(), best_path)
        print(f"--- Saved best model with Dice: {best_dice:.4f} ---")

    history.append({
        "epoch": epoch + 1,
        "train_loss": round(train_loss, 6),
        "train_dice": round(train_dice, 6),
        "val_dice": round(val_dice, 6),
        "val_iou": round(val_iou, 6),
        "best_dice": round(best_dice, 6),
    })

# Save final model
final_path = os.path.join(Config.CHECKPOINT_DIR, "segformer_polyp_final.pth")
torch.save(model.state_dict(), final_path)
print(f"\nTraining completed. Final model saved to {final_path}")

## 10. Save Training History

In [ ]:
history_path = os.path.join(Config.OUTPUT_DIR, "training_history.json")
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f"Training history saved to {history_path}")

## 11. Visualization

In [ ]:
# Plot training history
epochs = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
train_dices = [h["train_dice"] for h in history]
val_dices = [h["val_dice"] for h in history]
val_ious = [h["val_iou"] for h in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(epochs, train_losses, marker='o')
axes[0, 0].set_title('Training Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True)

# Train Dice
axes[0, 1].plot(epochs, train_dices, marker='o', label='Train')
axes[0, 1].plot(epochs, val_dices, marker='s', label='Val')
axes[0, 1].set_title('Dice Score')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Dice')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Val IoU
axes[1, 0].plot(epochs, val_ious, marker='o', color='green')
axes[1, 0].set_title('Validation IoU')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('IoU')
axes[1, 0].grid(True)

# Best Dice
best_dices = [h["best_dice"] for h in history]
axes[1, 1].plot(epochs, best_dices, marker='o', color='red')
axes[1, 1].set_title('Best Dice Score')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Best Dice')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(Config.OUTPUT_DIR, 'training_history.png'), dpi=100)
plt.show()

print(f"Training history plot saved.")

## 12. Test Predictions (Optional)

In [ ]:
# Load best model
best_model_path = os.path.join(Config.CHECKPOINT_DIR, "segformer_polyp_best.pth")
model.load_state_dict(torch.load(best_model_path, map_location=Config.DEVICE))
model.eval()

# Get a sample from validation set
sample_batch = next(iter(val_loader))
pixel_values = sample_batch["pixel_values"].to(Config.DEVICE)
labels = sample_batch["labels"].to(Config.DEVICE)

with torch.no_grad():
    outputs = model(pixel_values=pixel_values)
    logits = outputs.logits
    upsampled_logits = torch.nn.functional.interpolate(
        logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
    )
    predictions = torch.sigmoid(upsampled_logits) > 0.5

# Visualize
fig, axes = plt.subplots(4, 3, figsize=(15, 16))

for i in range(min(4, len(pixel_values))):
    # Original image
    img = pixel_values[i].cpu().numpy().transpose(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min())
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Image {i+1}')
    axes[i, 0].axis('off')
    
    # Ground truth
    axes[i, 1].imshow(labels[i].cpu().numpy(), cmap='gray')
    axes[i, 1].set_title(f'Ground Truth {i+1}')
    axes[i, 1].axis('off')
    
    # Prediction
    axes[i, 2].imshow(predictions[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 2].set_title(f'Prediction {i+1}')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(Config.OUTPUT_DIR, 'predictions_sample.png'), dpi=100)
plt.show()

print("Sample predictions visualized.")

## 13. Summary

In [ ]:
print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
print(f"Model: {Config.MODEL_NAME}")
print(f"Total Epochs: {Config.EPOCHS}")
print(f"Batch Size: {Config.BATCH_SIZE}")
print(f"Learning Rate: {Config.LEARNING_RATE}")
print(f"\nBest Validation Dice: {best_dice:.4f}")
print(f"\nCheckpoints saved to: {Config.CHECKPOINT_DIR}")
print(f"Training history saved to: {history_path}")
print(f"\nOutput directory: {Config.OUTPUT_DIR}")
print("="*50)